In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/shivamb/machine-predictive-maintenance-classification/predictive_maintenance.csv


In [2]:
import pandas as pd

df = pd.read_csv(
    "/kaggle/input/datasets/shivamb/machine-predictive-maintenance-classification/predictive_maintenance.csv"
)

df.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Target,Failure Type
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,No Failure
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,No Failure
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,No Failure
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,No Failure
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,No Failure


In [ ]:
# Predictive Maintenance — AI4I 2020 Dataset
# Machine learning and deep learning models for binary failure classification

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


# **Dataset Selection**

In [ ]:
df.shape

In [ ]:
df.info()


In [ ]:
df.describe()


In [ ]:
# Missing values
df.isnull().sum()


In [ ]:
# Duplicate rows
df.duplicated().sum()


In [ ]:
# Target class distribution
df["Target"].value_counts()


In [ ]:
# Failure type breakdown
df["Failure Type"].value_counts()


In [ ]:
# Product type distribution
df["Type"].value_counts()


In [ ]:
df.hist(figsize=(12, 8))
plt.show()


In [ ]:
numeric_cols = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
]

df[numeric_cols].plot(kind="box", figsize=(10, 6))
plt.show()


In [ ]:
corr = df.select_dtypes(include="number").corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap="coolwarm")
plt.show()


# **Data Preprocessing**

In [ ]:
# Remove unnecessary columns (identifiers, not predictive features)
df = df.drop(columns=["UDI", "Product ID"])

In [ ]:
# Encode categorical feature
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["Type"] = le.fit_transform(df["Type"])


In [ ]:
# "Failure Type" is dropped along with the target to prevent data leakage
X = df.drop(columns=["Target", "Failure Type"])
y = df["Target"]


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# **Model Training & Evaluation**

Seven models are trained and compared: Logistic Regression, Decision Tree, Random Forest, XGBoost, LightGBM, CatBoost, and an MLP (deep learning).

In [ ]:
import time

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, RocCurveDisplay
)

# XGBoost / LightGBM don't allow "[ ]" or "<" in column names
X_train.columns = X_train.columns.str.replace(r'[\[\]<]', '_', regex=True)
X_test.columns  = X_test.columns.str.replace(r'[\[\]<]', '_', regex=True)


In [ ]:
def evaluate_model(model_name, y_true, y_pred, y_prob, train_time):
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    f1   = f1_score(y_true, y_pred, zero_division=0)
    auc  = roc_auc_score(y_true, y_prob)
    cm   = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    print(f"\n{'='*45}")
    print(f"  {model_name}")
    print(f"{'='*45}")
    print(f"  Accuracy  : {acc:.4f}")
    print(f"  Precision : {prec:.4f}")
    print(f"  Recall    : {rec:.4f}")
    print(f"  F1-Score  : {f1:.4f}")
    print(f"  ROC-AUC   : {auc:.4f}")
    print(f"  Train Time: {train_time:.3f}s")
    print(f"\n  Confusion Matrix:")
    print(f"    TN={tn}  FP={fp}")
    print(f"    FN={fn}  TP={tp}")
    print(classification_report(y_true, y_pred,
                                target_names=["Healthy", "Failure"],
                                zero_division=0))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
                xticklabels=['Healthy', 'Failure'],
                yticklabels=['Healthy', 'Failure'],
                annot_kws={"size": 14})
    axes[0].set_title(f'{model_name}\nConfusion Matrix', fontsize=13, fontweight='bold')
    axes[0].set_xlabel('Predicted', fontsize=11)
    axes[0].set_ylabel('Actual', fontsize=11)

    RocCurveDisplay.from_predictions(y_true, y_prob, ax=axes[1],
                                     name=model_name, color='steelblue')
    axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1)
    axes[1].set_title(f'{model_name}\nROC Curve (AUC={auc:.4f})', fontsize=13, fontweight='bold')
    axes[1].set_xlabel('False Positive Rate', fontsize=11)
    axes[1].set_ylabel('True Positive Rate', fontsize=11)
    plt.tight_layout()
    plt.savefig(f"{model_name.replace(' ', '_')}_eval.png", dpi=150, bbox_inches='tight')
    plt.show()

    return {
        "Model"           : model_name,
        "Accuracy"        : round(acc,  4),
        "Precision"       : round(prec, 4),
        "Recall"          : round(rec,  4),
        "F1-Score"        : round(f1,   4),
        "ROC-AUC"         : round(auc,  4),
        "Training Time(s)": round(train_time, 3),
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
    }


In [ ]:
results = []

# 1. Logistic Regression (needs scaled data)
print("\nTraining Logistic Regression...")
lr = LogisticRegression(random_state=42, max_iter=1000)
start = time.time()
lr.fit(X_train_scaled, y_train)
results.append(evaluate_model("Logistic Regression", y_test,
    lr.predict(X_test_scaled),
    lr.predict_proba(X_test_scaled)[:, 1],
    time.time() - start))

# 2. Decision Tree
print("\nTraining Decision Tree...")
dt = DecisionTreeClassifier(random_state=42)
start = time.time()
dt.fit(X_train, y_train)
results.append(evaluate_model("Decision Tree", y_test,
    dt.predict(X_test),
    dt.predict_proba(X_test)[:, 1],
    time.time() - start))

# 3. Random Forest
print("\nTraining Random Forest...")
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
start = time.time()
rf.fit(X_train, y_train)
results.append(evaluate_model("Random Forest", y_test,
    rf.predict(X_test),
    rf.predict_proba(X_test)[:, 1],
    time.time() - start))

# 4. XGBoost
print("\nTraining XGBoost...")
xgb_model = XGBClassifier(random_state=42, eval_metric='logloss', verbosity=0)
start = time.time()
xgb_model.fit(X_train, y_train)
results.append(evaluate_model("XGBoost", y_test,
    xgb_model.predict(X_test),
    xgb_model.predict_proba(X_test)[:, 1],
    time.time() - start))

# 5. LightGBM
print("\nTraining LightGBM...")
lgbm_model = lgb.LGBMClassifier(random_state=42, verbosity=-1)
start = time.time()
lgbm_model.fit(X_train, y_train)
results.append(evaluate_model("LightGBM", y_test,
    lgbm_model.predict(X_test),
    lgbm_model.predict_proba(X_test)[:, 1],
    time.time() - start))

# 6. CatBoost
print("\nTraining CatBoost...")
cat_model = CatBoostClassifier(random_state=42, verbose=0)
start = time.time()
cat_model.fit(X_train, y_train)
results.append(evaluate_model("CatBoost", y_test,
    cat_model.predict(X_test),
    cat_model.predict_proba(X_test)[:, 1],
    time.time() - start))

# 7. MLP (Deep Learning)
print("\nTraining MLP (Deep Learning)...")
mlp_model = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    learning_rate_init=0.001,
    alpha=0.001,
    batch_size=32,
    max_iter=1000,
    early_stopping=True,
    validation_fraction=0.2,
    n_iter_no_change=5,
    random_state=42
)
start = time.time()
mlp_model.fit(X_train_scaled, y_train)
results.append(evaluate_model("MLP (Deep Learning)", y_test,
    mlp_model.predict(X_test_scaled),
    mlp_model.predict_proba(X_test_scaled)[:, 1],
    time.time() - start))


In [ ]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values("F1-Score", ascending=False).reset_index(drop=True)

print("=" * 85)
print("  ALL MODELS — FINAL COMPARISON (sorted by F1-Score)")
print("=" * 85)
display_cols = ["Model", "Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC", "Training Time(s)"]
print(results_df[display_cols].to_string(index=False))
print("=" * 85)

results_df.to_csv("all_models_results.csv", index=False)
print("\nSaved: all_models_results.csv")


In [ ]:
metrics = ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]
models  = results_df["Model"].tolist()
x       = np.arange(len(metrics))
n       = len(models)
width   = 0.11
colors  = ["#4C72B0", "#DD8452", "#55A868", "#C44E52",
           "#9467BD", "#8C564B", "#E377C2"]

fig, ax = plt.subplots(figsize=(16, 6))
for i, (model, color) in enumerate(zip(models, colors)):
    vals = results_df[results_df["Model"] == model][metrics].values[0]
    bars = ax.bar(x + i * width, vals, width, label=model, color=color, alpha=0.88)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
                f"{val:.3f}", ha='center', va='bottom', fontsize=6.5, fontweight='bold', rotation=90)

ax.set_xlabel("Metric", fontsize=12)
ax.set_ylabel("Score", fontsize=12)
ax.set_title("Model Performance Comparison", fontsize=14, fontweight='bold')
ax.set_xticks(x + width * (n / 2))
ax.set_xticklabels(metrics, fontsize=11)
ax.set_ylim(0, 1.15)
ax.legend(fontsize=9, loc='lower right')
ax.yaxis.grid(True, linestyle='--', alpha=0.6)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig("all_models_comparison.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: all_models_comparison.png")


# **Explainability (SHAP)**

XGBoost achieved the best overall F1-score and is used as the final selected model. SHAP is applied to it to explain which features drive its predictions.

In [ ]:
# !pip install shap -q   # uncomment on a fresh environment

import shap

shap.initjs()
explainer   = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)

# Beeswarm: feature impact distribution across all predictions
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test, show=False)
plt.title("SHAP Summary: What Drives Machine Failures?", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig("shap_beeswarm.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: shap_beeswarm.png")

# Bar: mean absolute SHAP value per feature
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test, plot_type="bar", show=False)
plt.title("SHAP Feature Importance (Mean |SHAP Value|)", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig("shap_bar.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: shap_bar.png")

# Waterfall: explanation for a single actual failure case
failure_indices = np.where(np.array(y_test) == 1)[0]
sample_idx      = failure_indices[0]
shap_explanation = explainer(X_test.iloc[[sample_idx]])

shap.plots.waterfall(shap_explanation[0], max_display=10, show=False)
fig = plt.gcf()
fig.suptitle("SHAP Waterfall: Why did this machine fail?", fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig("shap_waterfall.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: shap_waterfall.png")


In [ ]:
print("All 7 models trained and evaluated. XGBoost selected as the best-performing model (highest F1-score).")
